# Task 3 — Scratch Micro-Swin Clean-Slate Screen 2

This is a separate GPU experiment. It does not continue the E1–E10 CNN chain and it does not retrain Clean-Slate Screen 1. **Run All starts four fits:** Usage folds 0 and 4, then Gender folds 0 and 4. Completed matching folds are reused after a Colab disconnect.

## 1. Mount Drive and load the Task 3 branch

Drive stores the teacher ZIP, the historical anchors, the persistent run registry, and every new model artifact.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time
import zipfile

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "task-3-gender-usage-classification"
REPO_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
LOCAL_DATA_ZIP = Path("/content/task3-data.zip")
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"

def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)

In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab GPU runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)
if (REPO_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{REPO_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "switch", BRANCH], cwd=REPO_DIR)
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, REPO_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Repository ready:", REPO_DIR)
print("Branch:", BRANCH)
print("Commit:", commit)

## 2. Restore the teacher dataset on the Colab disk

Only the teacher dataset is extracted. Task 3 does not use the high-resolution dataset.

In [ ]:
def copy_teacher_zip_to_local_disk():
    if LOCAL_DATA_ZIP.is_file():
        try:
            with zipfile.ZipFile(LOCAL_DATA_ZIP) as existing:
                existing.infolist()
            print("Using the existing local ZIP copy:", LOCAL_DATA_ZIP)
            return
        except zipfile.BadZipFile:
            LOCAL_DATA_ZIP.unlink()

    partial = LOCAL_DATA_ZIP.with_suffix(".zip.partial")
    for attempt in range(1, 4):
        partial.unlink(missing_ok=True)
        try:
            if not DATA_ZIP.is_file():
                raise FileNotFoundError(f"Dataset archive not found: {DATA_ZIP}")
            expected_bytes = DATA_ZIP.stat().st_size
            print(
                f"Copying {expected_bytes / 1024**3:.2f} GiB from Drive to Colab "
                f"(attempt {attempt}/3)...",
                flush=True,
            )
            copied = 0
            next_report = 256 * 1024**2
            with DATA_ZIP.open("rb") as source, partial.open("wb") as target:
                while chunk := source.read(8 * 1024**2):
                    target.write(chunk)
                    copied += len(chunk)
                    if copied >= next_report:
                        print(f"  copied {copied / 1024**2:.0f} MiB", flush=True)
                        next_report += 256 * 1024**2
            if copied != expected_bytes:
                raise OSError(f"ZIP copy is incomplete: {copied} of {expected_bytes} bytes")
            partial.replace(LOCAL_DATA_ZIP)
            print("Local ZIP copy ready:", LOCAL_DATA_ZIP)
            return
        except (OSError, FileNotFoundError) as error:
            partial.unlink(missing_ok=True)
            if attempt == 3:
                raise RuntimeError(
                    "Google Drive disconnected three times while copying the teacher ZIP. "
                    "Reconnect Drive, then rerun this cell."
                ) from error
            print(f"Drive read failed: {error}. Remounting Drive...", flush=True)
            try:
                drive.flush_and_unmount()
            except Exception as unmount_error:
                print("Drive unmount warning:", unmount_error)
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
            time.sleep(2)


copy_teacher_zip_to_local_disk()

teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_suffixes = {".jpg", ".jpeg"}
image_dirs = (teacher_dir / "train/images_train", teacher_dir / "test/images_test")

with zipfile.ZipFile(LOCAL_DATA_ZIP) as archive:
    names = archive.namelist()
    unsafe = [name for name in names if Path(name).is_absolute() or ".." in Path(name).parts]
    if unsafe:
        raise RuntimeError("The dataset archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    if expected_images == 0:
        raise RuntimeError("The archive has no teacher images in data/raw/teacher.")
    current_images = sum(
        path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
    )
    if current_images != expected_images or not all(path.is_file() for path in required_files):
        print(f"Extracting {expected_images:,} teacher images...", flush=True)
        archive.extractall(REPO_DIR)
    else:
        print("Teacher data is already extracted; skipping.")

actual_images = sum(
    path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
)
missing_files = [str(path) for path in required_files if not path.is_file()]
if actual_images != expected_images or missing_files:
    raise RuntimeError(
        f"Dataset check failed: expected {expected_images:,} images, found {actual_images:,}; "
        f"missing files: {missing_files}"
    )
print(f"Teacher data ready: {actual_images:,} images")

os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
source_dir = str(REPO_DIR / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
(DRIVE_TASK_DIR / "results").mkdir(parents=True, exist_ok=True)
print("Drive output root:", DRIVE_TASK_DIR)

## 3. Verify the GPU screen without training

This builds both random-weight models and checks their output shapes, class order, teacher files, family-safe folds, runtime, and 7 GiB GPU limit. It creates no optimizer and takes no training step.

In [ ]:
from fashion.train.task3_micro_swin import check_micro_swin_screen_setup

screen_check = check_micro_swin_screen_setup(root=REPO_DIR, device_name="cuda")
if screen_check["optimizer_steps"] != 0 or screen_check["pretrained_weights_loaded"]:
    raise RuntimeError("The preflight unexpectedly trained or loaded weights.")
print("GPU:", screen_check["environment"]["gpu"])
print("Screen folds:", screen_check["screen_folds"])
for target, details in screen_check["targets"].items():
    print(target, details["input_view"], f'{details["parameter_count"]:,} parameters')

## 4. Frozen hypotheses and fair stopping rule

**Why this model is new.** Every learned E1–E10 candidate used convolution blocks. This model converts each 4×4 image patch with a linear layer, then mixes patches through local and shifted-window attention. It starts from random weights and contains no convolution layer or pretrained component.

**Gender hypothesis.** The foreground EDA showed that foreground-masked HOG was stronger than full-canvas HOG. A foreground-masked micro-Swin may combine child/adult, silhouette, colour, and accessory cues without relying as heavily on the catalogue background. It must still be rejected if the train–validation gap remains large or its matched score/classes fail the frozen screen gate.

**Usage hypothesis.** The EDA showed full-image HOG was stronger than foreground-only HOG for usage. A full-image micro-Swin may combine product parts and wider context that a flat CNN or the type-only rule misses. The literal `NA` and `Home` output classes remain present. Report both all-nine and without-`Home` macro-F1.

**Screen rule.** Run only canonical folds 0 and 4 with seed 2753. Keep the fixed final epoch; do not select a checkpoint from these outer validation folds. Advance only after reviewing score, per-class change, overfitting, robustness, runtime, and memory.

## 5. Train Usage scratch micro-Swin

This trains folds 0 and 4 only. Fold-only class weights are kept because Usage is extremely imbalanced. Artifacts are written directly to Drive after every epoch and completed fold.

In [ ]:
from fashion.train.task3_micro_swin import run_micro_swin_screen

usage_anchor = (
    DRIVE_TASK_DIR
    / "experiments/t3_usage_e8_translation/usage/aggregate/oof_predictions.csv"
)
if not usage_anchor.is_file():
    raise FileNotFoundError(f"Usage E8 anchor is missing: {usage_anchor}")
usage_micro_swin = run_micro_swin_screen(
    "usage",
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[LOCAL_REGISTRY],
    root=REPO_DIR,
    device_name="cuda",
    anchor_prediction_path=usage_anchor,
    reuse_completed=True,
)
{
    "metrics_path": usage_micro_swin["metrics_path"],
    "macro_f1": usage_micro_swin["metrics"]["macro_f1"],
    "macro_f1_without_home": usage_micro_swin["metrics"]["macro_f1_without_home"],
    "screen_gate": usage_micro_swin["metrics"]["screen_gate"],
}

## 6. Train Gender scratch micro-Swin

This trains the same frozen transformer family on the foreground-masked Gender input for folds 0 and 4. It is an independent model with its own head, normalization, checkpoint, and registry rows.

In [ ]:
gender_anchor = (
    DRIVE_TASK_DIR
    / "experiments/t3_gender_e9_semantic_filter/gender/aggregate/oof_predictions.csv"
)
if not gender_anchor.is_file():
    raise FileNotFoundError(f"Gender E9 anchor is missing: {gender_anchor}")
gender_micro_swin = run_micro_swin_screen(
    "gender",
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=[LOCAL_REGISTRY],
    root=REPO_DIR,
    device_name="cuda",
    anchor_prediction_path=gender_anchor,
    reuse_completed=True,
)
{
    "metrics_path": gender_micro_swin["metrics_path"],
    "macro_f1": gender_micro_swin["metrics"]["macro_f1"],
    "screen_gate": gender_micro_swin["metrics"]["screen_gate"],
}

## 7. Confirm the saved screen evidence

This only summarizes the two completed screen aggregates. It does not promote either model or start five-fold training.

In [ ]:
import pandas as pd

summary = pd.DataFrame(
    [
        {
            "target": target,
            "model_family": result["metrics"]["model_family"],
            "folds": result["metrics"]["validation_folds"],
            "macro_f1": result["metrics"]["macro_f1"],
            "fold_sd": result["metrics"]["fold_macro_f1_sample_sd"],
            "gate": result["metrics"]["screen_gate"]["status"],
            "metrics_path": result["metrics_path"],
        }
        for target, result in (
            ("usage", usage_micro_swin),
            ("gender", gender_micro_swin),
        )
    ]
)
display(summary)
print("Stop here. Review Screen 1 and Screen 2 before any five-fold promotion.")